In [1]:
# 방문 로그가 아닌 리뷰 기반 proxy이며, 실제 방문과 1:1 대응은 아니지만 시간대/요일의 상대적 분포를 보는 용도로 사용했다.

# 분석 가설명: H7. 일반 매장 대비 리저브 매장은 '주말·저녁’ 시간대의 비중이 더 높다.

In [4]:
# survey_response.csv의 데이터가 "각 매장 최신순 50개"로, 매장별 리뷰수 상이(30개 미만 존재) 및 날짜 최신쪽으로 몰려 있어, 
# row - level로 특정 매장이 결과를 지배할 위험 우려 발생.
# 고로, 분석 단위(관측치) 결정 필요
# 매장 단위로 해당 가설 분석 하기 

# 관측치 = store_id
# 각 메징 안에서 주말비중(weekend_share), 저녁비중(evening_share), 주말and 저녁(weekend_evening_share) 비중 비율 생성
# 해당 비율 분포를 리저브(63) vs 일반 매장(2050) 비교

# 리뷰 개수 적은 곳들의 비율 불안정 -> 최소 리뷰 수 컷(30개)이 필요

In [19]:
import pandas as pd
import numpy as np

# csv 로딩 함수(인코딩 자동)
def read_csv_guess(path):
    last = None
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last = e
    raise last

# 파일 로드
detail_path = r'C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data\store_review_detail.csv'
info_path = r'C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data\store_info.csv'

detail = read_csv_guess(detail_path)
info = read_csv_guess(info_path)

print("detail:", detail.shape)
print("detail columns:", detail.columns.tolist())

print("\ninfo:", info.shape)
print("info columns:", info.columns.tolist())

detail: (102472, 5)
detail columns: ['store_id', 'reviewer_id', 'visit_date', 'visit_count', 'visit_time']

info: (2113, 8)
info columns: ['store_id', 'store_name', 'store_region', 'store_type', 'store_address', 'store_latitude', 'store_longitude', 'store_open_date']


In [21]:
# 컬럼 값/ 점검 셀 확인
check_cols = ["store_id", "store_name", "store_type", "store_group", "visit_date", "visit_time", "visit_count", "day_type"]

print("[detail 컬럼 존재 여부]")
for c in check_cols:
    print(f"{c}: {c in detail.columns}")

print("\n[info 컬럼 존재 여부]")
for c in check_cols:
    print(f"{c}: {c in info.columns}")

for c in ["store_type", "store_group", "visit_time", "day_type"]:
    if c in detail.columns:
        print(f"\ndetail[{c}] unique sample:", detail[c].dropna().astype(str).unique()[:10])

for c in ["store_type", "store_group"]:
    if c in info.columns:
        print(f"\ninfo[{c}] unique sample:", info[c].dropna().astype(str).unique()[:10])

[detail 컬럼 존재 여부]
store_id: True
store_name: False
store_type: False
store_group: False
visit_date: True
visit_time: True
visit_count: True
day_type: False

[info 컬럼 존재 여부]
store_id: True
store_name: True
store_type: True
store_group: False
visit_date: False
visit_time: False
visit_count: False
day_type: False

detail[visit_time] unique sample: <StringArray>
['아침', '점심', '저녁']
Length: 3, dtype: str

info[store_type] unique sample: <StringArray>
['일반 매장', '리저브 매장']
Length: 2, dtype: str


In [42]:
# QC. 크롤링 데이터 누락 검토 (info에는 있는데 detail에는 없는 매장)
# 권장 위치: 파일 로드 직후 (detail/info 원본 기준)

import pandas as pd
import numpy as np

# -------------------------
# 0) 사본 사용 (원본 훼손 방지)
# -------------------------
detail_qc = detail.copy()
info_qc = info.copy()

# -------------------------
# 1) store_id 타입/공백 통일
# -------------------------
if "store_id" not in detail_qc.columns or "store_id" not in info_qc.columns:
    raise ValueError("detail/info 모두에 store_id 컬럼이 있어야 합니다.")

detail_qc["store_id"] = detail_qc["store_id"].astype(str).str.strip()
info_qc["store_id"] = info_qc["store_id"].astype(str).str.strip()

# 빈값 제거
detail_qc = detail_qc[detail_qc["store_id"].notna() & (detail_qc["store_id"] != "")]
info_qc = info_qc[info_qc["store_id"].notna() & (info_qc["store_id"] != "")]

# -------------------------
# 2) info 기준 매장유형 컬럼 통일 (store_type / store_group 대응)
# -------------------------
# info에 store_type이 있으면 그걸 쓰고, 없으면 store_group 사용
if "store_type" in info_qc.columns:
    type_col = "store_type"
elif "store_group" in info_qc.columns:
    type_col = "store_group"
else:
    raise ValueError("info에 store_type 또는 store_group 컬럼이 없습니다.")

info_qc[type_col] = info_qc[type_col].astype(str).str.strip()

# -------------------------
# 3) 유형 정규화 함수 (일반/리저브로 통일)
# -------------------------
def norm_group(x):
    x = str(x).strip().lower()
    # 리저브 판정
    if ("리저브" in x) or (x == "reserve"):
        return "리저브"
    # 일반 판정
    if ("일반" in x) or (x == "general"):
        return "일반"
    # 혹시 store_group이 general/reserve 말고 다른 값이면 기본 처리
    return np.nan

info_qc["store_group_norm"] = info_qc[type_col].apply(norm_group)

# 정규화 실패값 확인
unknown_group = info_qc.loc[info_qc["store_group_norm"].isna(), type_col].value_counts(dropna=False)
if len(unknown_group) > 0:
    print("⚠️ store_group 정규화 실패 값들:")
    print(unknown_group.head(20))

# 분석 가능한 값만 사용
info_qc_valid = info_qc[info_qc["store_group_norm"].isin(["일반", "리저브"])].copy()

# -------------------------
# 4) detail에 실제 존재하는 store_id 집합
# -------------------------
detail_store_ids = set(detail_qc["store_id"].unique())

# -------------------------
# 5) info 기준 리저브 / 일반 집합
# -------------------------
reserve_info_ids = set(
    info_qc_valid.loc[info_qc_valid["store_group_norm"] == "리저브", "store_id"].unique()
)

general_info_ids = set(
    info_qc_valid.loc[info_qc_valid["store_group_norm"] == "일반", "store_id"].unique()
)

# -------------------------
# 6) info에는 있는데 detail에는 없는 매장 찾기
# -------------------------
missing_reserve_ids = sorted(reserve_info_ids - detail_store_ids)
missing_general_ids = sorted(general_info_ids - detail_store_ids)

# -------------------------
# 7) 결과 출력
# -------------------------
print("=== [리저브] ===")
print("info 상 리저브 매장 수:", len(reserve_info_ids))
print("detail에 존재하는 리저브 store_id 수:", len(reserve_info_ids & detail_store_ids))
print("detail에 없는 리저브 매장 수:", len(missing_reserve_ids))
print("예시(앞 20개):", missing_reserve_ids[:20])

print("\n=== [일반] ===")
print("info 상 일반 매장 수:", len(general_info_ids))
print("detail에 존재하는 일반 store_id 수:", len(general_info_ids & detail_store_ids))
print("detail에 없는 일반 매장 수:", len(missing_general_ids))
print("예시(앞 20개):", missing_general_ids[:20])

# -------------------------
# 8) (선택) 누락 매장 목록 테이블로 보기
# -------------------------
# store_name이 info에 있으면 같이 출력
cols_show = ["store_id"]
if "store_name" in info_qc_valid.columns:
    cols_show.append("store_name")
cols_show.append("store_group_norm")

missing_df = info_qc_valid[info_qc_valid["store_id"].isin(missing_reserve_ids + missing_general_ids)][cols_show].drop_duplicates()

print("\n[누락 매장 목록 샘플]")
print(missing_df.head(20))

=== [리저브] ===
info 상 리저브 매장 수: 63
detail에 존재하는 리저브 store_id 수: 63
detail에 없는 리저브 매장 수: 0
예시(앞 20개): []

=== [일반] ===
info 상 일반 매장 수: 2050
detail에 존재하는 일반 store_id 수: 2050
detail에 없는 일반 매장 수: 0
예시(앞 20개): []

[누락 매장 목록 샘플]
Empty DataFrame
Columns: [store_id, store_name, store_group_norm]
Index: []


In [24]:
#1. store_review_detail 정리 

#1.1 visit_date 기존 문자열을 pd.to_datetime으로 변환하기(visit_date 파싱)
#날짜 변환 시도 (임시 컬럼에 저장하여 실패 데이터 식별)
# errors="coerce"를 사용해 변환 실패 시 NaT로 변환
detail["visit_date_cleaned"] = pd.to_datetime(detail["visit_date"], errors="coerce")

# [조회용] 날짜 파싱 실패 데이터 확인(중요 검문소)
# 이 failed_dates 호출 시 어떤 원본 데이터가 날짜로 안 바뀌었는지 확인 가능
failed_dates = detail.loc[detail["visit_date_cleaned"].isna(), "visit_date"].value_counts().head(20)
print("날짜 파싱 실패 데이터 상위:", failed_dates)

# [정제] 날짜 파싱 실패 제거
# NaT(실패)가 있는 행을 제거하고, 깨끗한 날짜로 기존 컬럼을 덮어쓰기
detail = detail.dropna(subset=["visit_date_cleaned"]).copy()
detail["visit_date"] = detail["visit_date_cleaned"]

# 임시 컬럼 삭제
detail.drop(columns=["visit_date_cleaned"], inplace=True)

print(f"\n✅ 날짜 정제 완료! 남은 데이터: {len(detail)}건")




#1.2 visit_time 카테고리 표준화(아침/점심/저녁-혹시 공백 및 섞여을 때 대비)
detail["visit_time"] = detail["visit_time"].astype(str).str.strip()

# 혹시 섞여있을 때 대비한 매핑
time_map = {
    "오전":"아침", "아침":"아침", "AM":"아침", "am":"아침",
    "점심":"점심", "오후":"점심",
    "저녁":"저녁", "밤":"저녁", "PM":"저녁", "pm":"저녁"
}

detail["visit_time_std"] = detail["visit_time"].map(time_map)

# 매핑 안 된 값 확인(중요)
# 이제 visit_time_std가 생성되었으니 조회가 가능합니다.
unknown_times = detail.loc[detail["visit_time_std"].isna(), "visit_time"].value_counts().head(20)
print("매핑 안 된 visit_time 상위:", unknown_times)

# 표준화 실패는 제거 (표준화된 값을 원본에 덮어쓰기)
detail = detail.dropna(subset=["visit_time_std"]).copy()
detail["visit_time"] = detail["visit_time_std"]

# 임시 컬럼 삭제
detail.drop(columns=["visit_time_std"], inplace=True)

print(f"\n✅ 시간 정제 완료! 남은 데이터: {len(detail)}건")



# 1.3 visit_count 타입 정리
# 숫자 변환 시도 (임시 컬럼 생성)
detail["visit_count_clean"] = pd.to_numeric(detail["visit_count"], errors="coerce")

# [조회용] 숫자 변환 실패(문자열 등) 데이터 확인
# 예를 들어 visit_count에 "모름"이나 공백이 적혀있던 경우 확인하기
unknown_counts = detail.loc[detail["visit_count_clean"].isna(), "visit_count"].value_counts().head(20)
print("숫자 변환 실패 visit_count 상위:", unknown_counts)

# [정제] 변환 실패(NaN) 제거 및 음수 제거
detail = detail.dropna(subset=["visit_count_clean"]).copy()
detail = detail[detail["visit_count_clean"] >= 0] # 음수 제거

# 최종 컬럼 확정 및 임시 컬럼 삭제
detail["visit_count"] = detail["visit_count_clean"].astype(int) # 정수형으로 변환
detail.drop(columns=["visit_count_clean"], inplace=True)



print(f"\n✅ 방문 횟수 정제 완료! 남은 데이터: {len(detail)}건")



날짜 파싱 실패 데이터 상위: Series([], Name: count, dtype: int64)

✅ 날짜 정제 완료! 남은 데이터: 102472건
매핑 안 된 visit_time 상위: Series([], Name: count, dtype: int64)

✅ 시간 정제 완료! 남은 데이터: 102472건
숫자 변환 실패 visit_count 상위: Series([], Name: count, dtype: int64)

✅ 방문 횟수 정제 완료! 남은 데이터: 102472건


In [25]:
#2. weekday/weekend 파생 변수 만들기
# weekday(월-금), weekend(토-일)

# 월 = 0 ~ 일 =6
detail["dow"] = detail["visit_date"].dt.dayofweek  # 월=0 ... 일=6
detail["day_type"] = np.where(detail["dow"].isin([5,6]), "주말", "주중")

# 날짜가 비어있거나 변환이 안 된 행은 dow가 NaN이 되고, 결과적으로 day_type 확인
failed_days = detail.loc[detail["day_type"].isna() | detail["dow"].isna()]

print(f"--- ⚠️ 요일 구분 실패 건수: {len(failed_days)}건 ---")

if len(failed_days) > 0:
    print("요일 구분 실패 데이터 샘플 (첫 5건):")
    print(failed_days[["visit_date", "dow", "day_type"]].head())
else:
    print("✅ 모든 데이터의 요일 구분이 완벽하게 완료되었습니다!")

# [정제] 혹시 모를 실패 데이터 제거
# (이미 앞에서 날짜 정제를 했으므로 0건이 나오는 것이 정상)
detail = detail.dropna(subset=["day_type"]).copy()

# 결과 확인
print("\n[CHECK] day_type별 데이터 분포")
print(detail["day_type"].value_counts())

--- ⚠️ 요일 구분 실패 건수: 0건 ---
✅ 모든 데이터의 요일 구분이 완벽하게 완료되었습니다!

[CHECK] day_type별 데이터 분포
day_type
주중    68774
주말    33698
Name: count, dtype: int64


In [27]:
import pandas as pd
import numpy as np
import os

# 3. info 붙이기 + store_group 생성 + 매장별 최신 50개 제한 + refined 저장

# -------------------------
# 3-1) 조인 키 타입 통일 (중요)
# -------------------------
detail["store_id"] = detail["store_id"].astype(str).str.strip()
info["store_id"] = info["store_id"].astype(str).str.strip()

# -------------------------
# 3-2) info에서 조인용 컬럼만 추출
# -------------------------
# store_type이 있으면 store_type 사용, 없으면 store_group 사용
if "store_type" in info.columns:
    info_sub = info[["store_id", "store_type"]].drop_duplicates("store_id").copy()
    info_sub["store_type"] = info_sub["store_type"].astype(str).str.strip()
elif "store_group" in info.columns:
    info_sub = info[["store_id", "store_group"]].drop_duplicates("store_id").copy()
    info_sub["store_group"] = info_sub["store_group"].astype(str).str.strip()
else:
    raise ValueError("info 파일에 store_type 또는 store_group 컬럼이 없습니다.")

# -------------------------
# 3-3) detail + info 매칭
# -------------------------
store_review_refined = detail.merge(
    info_sub, on="store_id", how="left", indicator="_merge_flag"
)

# 매핑 실패 확인
failed_mapping = store_review_refined.loc[
    store_review_refined["_merge_flag"] == "left_only", "store_id"
].unique()

print(f"⚠️ 매핑 실패한 store_id 개수: {len(failed_mapping)}개")
if len(failed_mapping) > 0:
    print("매핑 실패 ID 샘플:", failed_mapping[:10])

# 매핑 성공만 남기기
store_review_refined = store_review_refined[
    store_review_refined["_merge_flag"] == "both"
].copy()

# -------------------------
# 3-4) store_group 생성/통일 (최종: 일반 / 리저브)
# -------------------------
# 케이스 1) info에서 store_type 받아온 경우
if "store_type" in store_review_refined.columns:
    store_review_refined["store_type"] = store_review_refined["store_type"].astype(str).str.strip()

    # 기존 한글/영문 혼재 대응
    store_review_refined["store_group"] = np.where(
        store_review_refined["store_type"].str.contains("리저브|reserve", case=False, na=False),
        "리저브",
        "일반"
    )

# 케이스 2) info에서 store_group 받아온 경우
elif "store_group" in store_review_refined.columns:
    store_review_refined["store_group"] = store_review_refined["store_group"].astype(str).str.strip().replace({
        "general": "일반",
        "reserve": "리저브",
        "일반 매장": "일반",
        "리저브 매장": "리저브",
        "일반": "일반",
        "리저브": "리저브",
    })

print("\n[CHECK] 매장 그룹 분포")
print(store_review_refined["store_group"].value_counts(dropna=False))

# -------------------------
# 3-5) 매장별 최신 리뷰 50개 제한
# (50개 초과 매장만 잘리고, 나머지는 그대로 유지)
# -------------------------
print("\n[작업 시작] 매장별 리뷰 개수 제한 (최대 50개)")

# 혹시 셀3에서 이미 datetime 변환했더라도 안전하게 한 번 더 체크
store_review_refined["visit_date"] = pd.to_datetime(
    store_review_refined["visit_date"], errors="coerce"
)

# 날짜 없는 행 제거
before_cnt = len(store_review_refined)
store_review_refined = store_review_refined.dropna(subset=["visit_date"]).copy()
after_cnt = len(store_review_refined)
print(f"날짜 변환 실패로 제거된 행: {before_cnt - after_cnt}건")

# 정렬 + 매장별 최신 50개 유지
store_review_refined = store_review_refined.sort_values(
    by=["store_id", "visit_date"], ascending=[True, False]
)

store_review_refined = (
    store_review_refined.groupby("store_id", group_keys=False)
    .head(50)
    .reset_index(drop=True)
)

# 확인: 매장별 최대 개수 체크
max_reviews = store_review_refined.groupby("store_id").size().max()
print(f"✅ 제한 완료! 최종 분석 데이터 건수: {len(store_review_refined)}건")
print(f"✅ 매장별 최대 리뷰 수(확인용): {max_reviews}개")  # 50 이하여야 정상

# 임시 컬럼 삭제
store_review_refined.drop(columns=["_merge_flag"], inplace=True, errors="ignore")

# -------------------------
# 3-6) 저장
# -------------------------
save_path = r"C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data"

if not os.path.exists(save_path):
    os.makedirs(save_path)

save_file = os.path.join(save_path, "store_review_refined.csv")
store_review_refined.to_csv(save_file, index=False, encoding="utf-8-sig")

print(f"✅ 저장 완료 -> {save_file}")

⚠️ 매핑 실패한 store_id 개수: 0개

[CHECK] 매장 그룹 분포
store_group
일반     99342
리저브     3130
Name: count, dtype: int64

[작업 시작] 매장별 리뷰 개수 제한 (최대 50개)
날짜 변환 실패로 제거된 행: 0건
✅ 제한 완료! 최종 분석 데이터 건수: 102342건
✅ 매장별 최대 리뷰 수(확인용): 50개
✅ 저장 완료 -> C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data\store_review_refined.csv


In [36]:
# 4. 분석용 refined 생성 + H7/EDA 파생변수 만들기

# 전처리 완료본을 분석용 이름으로 복사
refined = store_review_refined.copy()

# -------------------------
# (선택) 값 최종 통일 검문소
# -------------------------
# day_type: 혹시 영어값이 남아있으면 한글로 통일
refined["day_type"] = refined["day_type"].astype(str).str.strip().replace({
    "weekday": "주중",
    "weekend": "주말",
    "주중": "주중",
    "주말": "주말"
})

# visit_time: 혹시 잔여 표현 있으면 한 번 더 통일
refined["visit_time"] = refined["visit_time"].astype(str).str.strip().replace({
    "오전": "아침", "AM": "아침", "am": "아침", "morning": "아침",
    "오후": "점심", "lunch": "점심", "afternoon": "점심",
    "밤": "저녁", "PM": "저녁", "pm": "저녁", "dinner": "저녁", "evening": "저녁",
    "아침": "아침", "점심": "점심", "저녁": "저녁"
})

# store_group: 혹시 영문/긴표현 섞여있으면 한 번 더 통일
refined["store_group"] = refined["store_group"].astype(str).str.strip().replace({
    "general": "일반",
    "reserve": "리저브",
    "일반 매장": "일반",
    "리저브 매장": "리저브",
    "일반": "일반",
    "리저브": "리저브"
})

# 핵심 컬럼 결측 제거
refined = refined.dropna(subset=["store_id", "visit_date", "visit_time", "visit_count", "day_type", "store_group"]).copy()

# 분석 대상만 남기기
refined = refined[refined["store_group"].isin(["일반", "리저브"])].copy()
refined = refined[refined["day_type"].isin(["주중", "주말"])].copy()
refined = refined[refined["visit_time"].isin(["아침", "점심", "저녁"])].copy()

# -------------------------
# H7 / EDA 파생변수 (row-level)
# -------------------------
# 기본 축
refined["is_weekday"] = (refined["day_type"] == "주중").astype(int)
refined["is_weekend"] = (refined["day_type"] == "주말").astype(int)

refined["is_morning"] = (refined["visit_time"] == "아침").astype(int)
refined["is_lunch"] = (refined["visit_time"] == "점심").astype(int)
refined["is_evening"] = (refined["visit_time"] == "저녁").astype(int)

# 6개 세부 조합 (주중/주말 × 아침/점심/저녁)
refined["is_weekday_morning"] = ((refined["day_type"] == "주중") & (refined["visit_time"] == "아침")).astype(int)
refined["is_weekday_lunch"]   = ((refined["day_type"] == "주중") & (refined["visit_time"] == "점심")).astype(int)
refined["is_weekday_evening"] = ((refined["day_type"] == "주중") & (refined["visit_time"] == "저녁")).astype(int)

refined["is_weekend_morning"] = ((refined["day_type"] == "주말") & (refined["visit_time"] == "아침")).astype(int)
refined["is_weekend_lunch"]   = ((refined["day_type"] == "주말") & (refined["visit_time"] == "점심")).astype(int)
refined["is_weekend_evening"] = ((refined["day_type"] == "주말") & (refined["visit_time"] == "저녁")).astype(int)


# 확인 출력
print("✅ refined 준비 완료:", refined.shape)
print("\n[store_group 분포]")
print(refined["store_group"].value_counts(dropna=False))
print("\n[day_type 분포]")
print(refined["day_type"].value_counts(dropna=False))
print("\n[visit_time 분포]")
print(refined["visit_time"].value_counts(dropna=False))

print("\n[파생변수 평균(=전체 비중)]")
print(
    refined[[
        "is_weekday", "is_weekend",
        "is_morning", "is_lunch", "is_evening",
        "is_weekday_morning","is_weekday_lunch", "is_weekday_evening",
        "is_weekend_morning","is_weekend_lunch", "is_weekend_evening"

    ]].mean().mul(100).round(1).astype(str) + "%"
)



# 파일 저장하기
save_path = r"C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data"
save_file = os.path.join(save_path, "refined_h7.csv")  # 파일명은 원하는 걸로 변경 가능

# 폴더 없으면 생성
os.makedirs(save_path, exist_ok=True)

# 저장 (한글 깨짐 방지)
refined.to_csv(save_file, index=False, encoding="utf-8-sig")

print(f"✅ 저장 완료: {save_file}")

✅ refined 준비 완료: (102342, 19)

[store_group 분포]
store_group
일반     99212
리저브     3130
Name: count, dtype: int64

[day_type 분포]
day_type
주중    68698
주말    33644
Name: count, dtype: int64

[visit_time 분포]
visit_time
점심    62716
저녁    23467
아침    16159
Name: count, dtype: int64

[파생변수 평균(=전체 비중)]
is_weekday            67.1%
is_weekend            32.9%
is_morning            15.8%
is_lunch              61.3%
is_evening            22.9%
is_weekday_morning    11.5%
is_weekday_lunch      40.2%
is_weekday_evening    15.5%
is_weekend_morning     4.3%
is_weekend_lunch      21.1%
is_weekend_evening     7.4%
dtype: str
✅ 저장 완료: C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data\refined_h7.csv


In [37]:
#4. 리뷰 수 매장마다 상이 문제 처리 (데이터 공정성 처리)
# 매장별 최신 리뷰 50개 크롤링하였으나, 
# 총 리뷰 개수 미달로 각 어떤 매장은 30개 미만, 어떤 매장은 50개
# 이 상태로 전체행(row) 기준으로 합계를 내면:
# 리뷰 많은 매장이 더 큰 영향을 가짐(편향)
# 특히 일반 매장은 수가 압도적으로 많아서 “그냥 일반의 패턴”이 되어버릴 수 있음
# 그래서 해결은 2단계로 최종 진행

# 해결책 1단계: “매장 단위 비율(share)”로 표준화
# 각 매장 1개 표본으로 만들고, 매장 내에서 비율을 계산하면 리뷰 수 달라도 공정해짐.


In [38]:
# B. refined -> refined_30 (민감도 분석용: 매장별 리뷰 30개 이상)

# 매장 식별키
store_key = "store_id"

# 매장별 리뷰 수 계산
store_counts = refined.groupby(store_key).size().rename("n_reviews").reset_index()

# 붙이기
if "n_reviews" in refined.columns:
    refined = refined.drop(columns=["n_reviews"])
refined = refined.merge(store_counts, on=store_key, how="left")

# 30개 이상만
refined_30 = refined[refined["n_reviews"] >= 30].copy()

print("✅ refined_30 완료")
print("refined   :", refined.shape)
print("refined_30:", refined_30.shape)

print("\n[매장 수]")
print("전체:", refined[store_key].nunique())
print("30컷:", refined_30[store_key].nunique())

print("\n[검문소] refined_30 매장별 최소 리뷰 수")
print(refined_30.groupby(store_key).size().min())

✅ refined_30 완료
refined   : (102342, 20)
refined_30: (101373, 20)

[매장 수]
전체: 2113
30컷: 2038

[검문소] refined_30 매장별 최소 리뷰 수
30


In [39]:
# C. 저장 (추천)
import os

save_path = r"C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data"
os.makedirs(save_path, exist_ok=True)

store_review_refined.to_csv(os.path.join(save_path, "store_review_refined.csv"), index=False, encoding="utf-8-sig")
refined.to_csv(os.path.join(save_path, "refined_h7.csv"), index=False, encoding="utf-8-sig")
refined_30.to_csv(os.path.join(save_path, "refined_h7_30.csv"), index=False, encoding="utf-8-sig")

print("✅ 저장 완료")

✅ 저장 완료


In [44]:
# QC. 크롤링 데이터 누락 검토 (info에는 있는데 detail에는 없는 매장)
# 권장 위치: 파일 로드 직후 (detail/info 원본 기준)

import pandas as pd
import numpy as np

# -------------------------
# 0) 사본 사용 (원본 훼손 방지)
# -------------------------
detail_qc = detail.copy()
info_qc = info.copy()

# -------------------------
# 1) store_id 타입/공백 통일
# -------------------------
if "store_id" not in detail_qc.columns or "store_id" not in info_qc.columns:
    raise ValueError("detail/info 모두에 store_id 컬럼이 있어야 합니다.")

detail_qc["store_id"] = detail_qc["store_id"].astype(str).str.strip()
info_qc["store_id"] = info_qc["store_id"].astype(str).str.strip()

# 빈값 제거
detail_qc = detail_qc[detail_qc["store_id"].notna() & (detail_qc["store_id"] != "")]
info_qc = info_qc[info_qc["store_id"].notna() & (info_qc["store_id"] != "")]

# -------------------------
# 2) info 기준 매장유형 컬럼 통일 (store_type / store_group 대응)
# -------------------------
# info에 store_type이 있으면 그걸 쓰고, 없으면 store_group 사용
if "store_type" in info_qc.columns:
    type_col = "store_type"
elif "store_group" in info_qc.columns:
    type_col = "store_group"
else:
    raise ValueError("info에 store_type 또는 store_group 컬럼이 없습니다.")

info_qc[type_col] = info_qc[type_col].astype(str).str.strip()

# -------------------------
# 3) 유형 정규화 함수 (일반/리저브로 통일)
# -------------------------
def norm_group(x):
    x = str(x).strip().lower()
    # 리저브 판정
    if ("리저브" in x) or (x == "reserve"):
        return "리저브"
    # 일반 판정
    if ("일반" in x) or (x == "general"):
        return "일반"
    # 혹시 store_group이 general/reserve 말고 다른 값이면 기본 처리
    return np.nan

info_qc["store_group_norm"] = info_qc[type_col].apply(norm_group)

# 정규화 실패값 확인
unknown_group = info_qc.loc[info_qc["store_group_norm"].isna(), type_col].value_counts(dropna=False)
if len(unknown_group) > 0:
    print("⚠️ store_group 정규화 실패 값들:")
    print(unknown_group.head(20))

# 분석 가능한 값만 사용
info_qc_valid = info_qc[info_qc["store_group_norm"].isin(["일반", "리저브"])].copy()

# -------------------------
# 4) detail에 실제 존재하는 store_id 집합
# -------------------------
detail_store_ids = set(detail_qc["store_id"].unique())

# -------------------------
# 5) info 기준 리저브 / 일반 집합
# -------------------------
reserve_info_ids = set(
    info_qc_valid.loc[info_qc_valid["store_group_norm"] == "리저브", "store_id"].unique()
)

general_info_ids = set(
    info_qc_valid.loc[info_qc_valid["store_group_norm"] == "일반", "store_id"].unique()
)

# -------------------------
# 6) info에는 있는데 detail에는 없는 매장 찾기
# -------------------------
missing_reserve_ids = sorted(reserve_info_ids - detail_store_ids)
missing_general_ids = sorted(general_info_ids - detail_store_ids)

# -------------------------
# 7) 결과 출력
# -------------------------
print("=== [리저브] ===")
print("info 상 리저브 매장 수:", len(reserve_info_ids))
print("detail에 존재하는 리저브 store_id 수:", len(reserve_info_ids & detail_store_ids))
print("detail에 없는 리저브 매장 수:", len(missing_reserve_ids))
print("예시(앞 20개):", missing_reserve_ids[:20])

print("\n=== [일반] ===")
print("info 상 일반 매장 수:", len(general_info_ids))
print("detail에 존재하는 일반 store_id 수:", len(general_info_ids & detail_store_ids))
print("detail에 없는 일반 매장 수:", len(missing_general_ids))
print("예시(앞 20개):", missing_general_ids[:20])

# -------------------------
# 8) (선택) 누락 매장 목록 테이블로 보기
# -------------------------
# (선택) 누락 매장 목록은 있을 때만 출력
total_missing = len(missing_reserve_ids) + len(missing_general_ids)

if total_missing > 0:
    cols_show = ["store_id"]
    if "store_name" in info_qc_valid.columns:
        cols_show.append("store_name")
    cols_show.append("store_group_norm")

    missing_df = info_qc_valid[
        info_qc_valid["store_id"].isin(missing_reserve_ids + missing_general_ids)
    ][cols_show].drop_duplicates()

    print("\n[누락 매장 목록 샘플]")
    print(missing_df.head(20))
else:
    print("\n✅ 누락 매장 없음 (목록 출력 생략)")

=== [리저브] ===
info 상 리저브 매장 수: 63
detail에 존재하는 리저브 store_id 수: 63
detail에 없는 리저브 매장 수: 0
예시(앞 20개): []

=== [일반] ===
info 상 일반 매장 수: 2050
detail에 존재하는 일반 store_id 수: 2050
detail에 없는 일반 매장 수: 0
예시(앞 20개): []

✅ 누락 매장 없음 (목록 출력 생략)
